# 13 — K-Dwarf Subpopulation Performance

Repeat the grouped validation on Notebook 09's K-dwarf temperature range and inspect performance across predefined subgroups.

**Reproducibility note:** This notebook follows Notebook 03's primary sample definition and Random Forest pipeline. It does not overwrite or modify any earlier notebook. Run cells from top to bottom in Kaggle or Jupyter.

In [4]:
import os, glob, warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, precision_score, recall_score, f1_score, log_loss
)

# Match Notebook 03's primary population and modeling settings.
RADIUS_MIN, RADIUS_MAX = 1.0, 4.0
TEFF_MIN, TEFF_MAX = 4700, 6500
N_SPLITS = 5
THRESHOLD = 0.50
RANDOM_STATE = 42
N_ESTIMATORS = 400

# Set this manually if auto-detection finds zero or multiple candidate CSV files.
DATA_PATH = None  # e.g. "/kaggle/input/your-dataset/cumulative.csv"

def locate_koi_csv():
    if DATA_PATH is not None:
        if not os.path.exists(DATA_PATH):
            raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}")
        return DATA_PATH
    candidates = sorted(
        p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)
        if ("cumulative" in os.path.basename(p).lower()
            or "koi" in os.path.basename(p).lower())
    )
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "No KOI cumulative CSV found under /kaggle/input. "
            "Attach the same catalog used in Notebook 03 and set DATA_PATH manually."
        )
    print("Possible KOI CSV files:")
    for p in candidates:
        print(" -", p)
    raise ValueError("Multiple possible KOI CSVs found. Set DATA_PATH explicitly.")

def load_main_sample():
    path = locate_koi_csv()
    print("Using catalog:", path)
    raw = pd.read_csv(path, comment="#", low_memory=False)
    required = {"kepid", "koi_disposition", "koi_prad", "koi_steff"}
    missing = required - set(raw.columns)
    if missing:
        raise KeyError(f"Required columns missing: {sorted(missing)}")
    for col in ["kepid", "koi_prad", "koi_steff"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    catalog = raw[
        raw["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE", "CANDIDATE"])
        & raw["kepid"].notna()
        & raw["koi_prad"].between(RADIUS_MIN, RADIUS_MAX, inclusive="both")
        & raw["koi_steff"].between(TEFF_MIN, TEFF_MAX, inclusive="both")
    ].copy()
    catalog["group"] = catalog["kepid"].astype("int64")
    labeled = catalog[
        catalog["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
    ].copy()
    labeled["label"] = (labeled["koi_disposition"] == "CONFIRMED").astype(int)
    candidate_features = [
        "koi_period", "koi_time0bk", "koi_impact", "koi_duration",
        "koi_depth", "koi_prad", "koi_teq", "koi_insol",
        "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"
    ]
    features = [c for c in candidate_features if c in labeled.columns]
    if len(features) < 3:
        raise ValueError(f"Fewer than 3 Notebook 03 features available: {features}")
    X = labeled[features].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    drop = [c for c in X if X[c].notna().sum() == 0 or X[c].nunique(dropna=True) <= 1]
    if drop:
        print("Dropping all-missing/constant features:", drop)
        X = X.drop(columns=drop)
    y = labeled["label"].to_numpy(dtype=int)
    groups = labeled["group"].to_numpy()
    if len(np.unique(y)) != 2:
        raise ValueError("Both CONFIRMED and FALSE POSITIVE labels are required.")
    if labeled.groupby("label")["group"].nunique().min() < N_SPLITS:
        raise ValueError("Too few distinct host stars in at least one class for 5 folds.")
    print(f"Filtered catalog rows: {len(catalog):,}")
    print(f"Resolved labeled rows: {len(labeled):,}")
    print(f"Unique resolved host stars: {len(np.unique(groups)):,}")
    print("Class counts (0=false positive, 1=confirmed):")
    print(pd.Series(y).value_counts().sort_index())
    print("Features:", list(X.columns))
    return path, raw, catalog, labeled, X, y, groups

def build_model(seed=RANDOM_STATE):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        ))
    ])

def make_oof(X, y, groups, grouped=True, seed=RANDOM_STATE, n_splits=N_SPLITS):
    splitter = (
        StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        if grouped else
        StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    )
    probs = np.full(len(y), np.nan, dtype=float)
    folds = np.full(len(y), -1, dtype=int)
    split_iter = splitter.split(X, y, groups) if grouped else splitter.split(X, y)
    for fold, (tr, te) in enumerate(split_iter):
        if np.unique(y[tr]).size != 2 or np.unique(y[te]).size != 2:
            raise ValueError(
                f"Fold {fold+1} has a single class. Inspect group distribution; "
                "do not silently report fold-level class metrics."
            )
        if grouped and set(groups[tr]).intersection(set(groups[te])):
            raise RuntimeError(f"Host-star overlap in fold {fold+1}.")
        model = build_model(seed=seed)
        model.fit(X.iloc[tr], y[tr])
        probs[te] = model.predict_proba(X.iloc[te])[:, 1]
        folds[te] = fold
    if np.isnan(probs).any() or (folds < 0).any():
        raise RuntimeError("Every row must receive exactly one out-of-fold prediction.")
    return probs, folds

def metric_dict(y, p, threshold=THRESHOLD):
    pred = (p >= threshold).astype(int)
    return {
        "ROC_AUC": roc_auc_score(y, p),
        "average_precision": average_precision_score(y, p),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "Brier_score_lower_is_better": brier_score_loss(y, p),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "threshold": threshold
    }

def save_csv(df, filename):
    os.makedirs("/kaggle/working", exist_ok=True)
    path = os.path.join("/kaggle/working", filename)
    df.to_csv(path, index=False)
    print("Saved:", path)

## Aim
Apply the same Notebook 03 model and grouped validation to the narrower K-dwarf sample used in Notebook 09: 1–4 Earth radii and stellar effective temperature 3,900–5,200 K. Then compare model performance across prespecified temperature, radius, and orbital-period slices where both labels are represented.

In [2]:
# K-dwarf-specific sample: intentionally overrides only the stellar-temperature bounds.
K_DWARF_TEMPERATURE_MIN = 3900
K_DWARF_TEMPERATURE_MAX = 5200
path = locate_koi_csv()
print("Using catalog:", path)
raw = pd.read_csv(path, comment="#", low_memory=False)
required = {"kepid", "koi_disposition", "koi_prad", "koi_steff"}
missing = required - set(raw.columns)
if missing:
    raise KeyError(f"Required columns missing: {sorted(missing)}")
for col in ["kepid", "koi_prad", "koi_steff"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

population = raw[
    raw["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE", "CANDIDATE"])
    & raw["kepid"].notna()
    & raw["koi_prad"].between(1.0, 4.0, inclusive="both")
    & raw["koi_steff"].between(K_DWARF_TEMPERATURE_MIN, K_DWARF_TEMPERATURE_MAX, inclusive="both")
].copy()
population["group"] = population["kepid"].astype("int64")
labeled = population[
    population["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
].copy()
labeled["label"] = (labeled["koi_disposition"] == "CONFIRMED").astype(int)

candidate_features = [
    "koi_period", "koi_time0bk", "koi_impact", "koi_duration",
    "koi_depth", "koi_prad", "koi_teq", "koi_insol",
    "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"
]
features = [c for c in candidate_features if c in labeled.columns]
X = labeled[features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
drop = [c for c in X if X[c].notna().sum() == 0 or X[c].nunique(dropna=True) <= 1]
X = X.drop(columns=drop)
y = labeled["label"].to_numpy(dtype=int)
groups = labeled["group"].to_numpy()

if len(np.unique(y)) != 2:
    raise ValueError("K-dwarf resolved sample must contain both labels.")
if labeled.groupby("label")["group"].nunique().min() < N_SPLITS:
    raise ValueError("Too few distinct K-dwarf host stars in one class for five grouped folds.")

print("K-dwarf population rows including candidates:", len(population))
print("Resolved labeled rows:", len(labeled))
print("Unique resolved host stars:", len(np.unique(groups)))
print("Candidates excluded from supervised model:", int((population["koi_disposition"] == "CANDIDATE").sum()))
print("Features:", list(X.columns))

grouped_p, grouped_fold = make_oof(X, y, groups, grouped=True, seed=RANDOM_STATE)
overall = pd.DataFrame([{
    "population": "K dwarfs 3900-5200 K; 1-4 Earth radii",
    "n_resolved_rows": len(y), "n_host_stars": len(np.unique(groups)),
    "n_confirmed_rows": int(y.sum()), "n_false_positive_rows": int((y == 0).sum()),
    **metric_dict(y, grouped_p)
}])
display(overall.round(4))
save_csv(overall, "13_k_dwarf_overall_metrics.csv")

Using catalog: /kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv
K-dwarf population rows including candidates: 852
Resolved labeled rows: 676
Unique resolved host stars: 520
Candidates excluded from supervised model: 176
Features: ['koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag']


,population,n_resolved_rows,n_host_stars,n_confirmed_rows,n_false_positive_rows,ROC_AUC,average_precision,balanced_accuracy,Brier_score_lower_is_better,precision,recall,F1,threshold
0,K dwarfs 3900-5200 K; 1-4 Earth radii,676,520,507,169,0.8859,0.9514,0.7446,0.1116,0.8631,0.9329,0.8967,0.5


Saved: /kaggle/working/13_k_dwarf_overall_metrics.csv


In [3]:
# Build subgroup labels using measured catalog values. These are diagnostic slices,
# not occurrence-rate bins. Report support and skip unstable one-class slices.
diagnostic = labeled.copy().reset_index(drop=True)
diagnostic["true_label"] = y
diagnostic["grouped_oof_probability"] = grouped_p

diagnostic["temperature_band_K"] = pd.cut(
    diagnostic["koi_steff"], bins=[3900, 4300, 4700, 5200],
    labels=["3900-4300", "4300-4700", "4700-5200"],
    include_lowest=True
)
diagnostic["radius_band_Re"] = pd.cut(
    diagnostic["koi_prad"], bins=[1, 2, 3, 4],
    labels=["1-2", "2-3", "3-4"], include_lowest=True
)
if "koi_period" in diagnostic.columns:
    period = pd.to_numeric(diagnostic["koi_period"], errors="coerce")
    diagnostic["period_band_days"] = pd.cut(
        period, bins=[0, 10, 30, np.inf],
        labels=["<=10", "10-30", ">30"], include_lowest=True
    )

subgroup_rows = []
for col in ["temperature_band_K", "radius_band_Re", "period_band_days"]:
    if col not in diagnostic.columns:
        continue
    for band, part in diagnostic.groupby(col, observed=False):
        if pd.isna(band) or len(part) == 0:
            continue
        row = {
            "dimension": col, "band": str(band),
            "n_rows": len(part), "n_host_stars": part["group"].nunique(),
            "n_confirmed": int(part["true_label"].sum()),
            "n_false_positive": int((part["true_label"] == 0).sum())
        }
        if part["true_label"].nunique() == 2:
            row.update(metric_dict(
                part["true_label"].to_numpy(),
                part["grouped_oof_probability"].to_numpy()
            ))
            row["metric_status"] = "computed; interpret with support counts"
        else:
            row["metric_status"] = "not computed: only one class in this slice"
        subgroup_rows.append(row)

subgroups = pd.DataFrame(subgroup_rows)
display(subgroups.round(4))
save_csv(subgroups, "13_k_dwarf_subgroup_metrics.csv")
save_csv(diagnostic[[
    "kepid", "koi_disposition", "koi_prad", "koi_steff",
    "true_label", "grouped_oof_probability"
] + (["koi_period"] if "koi_period" in diagnostic.columns else [])],
"13_k_dwarf_oof_predictions.csv")

,dimension,band,n_rows,n_host_stars,n_confirmed,n_false_positive,ROC_AUC,average_precision,balanced_accuracy,Brier_score_lower_is_better,precision,recall,F1,threshold,metric_status
0,temperature_band_K,3900-4300,93,61,77,16,0.9513,0.9892,0.7683,0.0703,0.9146,0.9740,0.9434,0.5,computed; interpret with support counts
1,temperature_band_K,4300-4700,118,98,77,41,0.8489,0.8934,0.7667,0.1432,0.8111,0.9481,0.8743,0.5,computed; interpret with support counts
2,temperature_band_K,4700-5200,465,361,353,112,0.8814,0.9512,0.7327,0.1118,0.8644,0.9207,0.8916,0.5,computed; interpret with support counts
3,radius_band_Re,1-2,373,315,270,103,0.8763,0.9440,0.7524,0.1214,0.8527,0.9222,0.8861,0.5,computed; interpret with support counts
4,radius_band_Re,2-3,246,215,193,53,0.9034,0.9606,0.7220,0.0948,0.8720,0.9534,0.9109,0.5,computed; interpret with support counts
5,radius_band_Re,3-4,57,53,44,13,0.8654,0.9563,0.7622,0.1195,0.8889,0.9091,0.8989,0.5,computed; interpret with support counts
6,period_band_days,<=10,343,288,259,84,0.8971,0.9612,0.7551,0.1064,0.8745,0.9151,0.8943,0.5,computed; interpret with support counts
7,period_band_days,10-30,172,156,150,22,0.8333,0.9571,0.5876,0.0812,0.8922,0.9933,0.9401,0.5,computed; interpret with support counts
8,period_band_days,>30,161,155,98,63,0.8477,0.8919,0.7613,0.1549,0.7909,0.8878,0.8365,0.5,computed; interpret with support counts


Saved: /kaggle/working/13_k_dwarf_subgroup_metrics.csv
Saved: /kaggle/working/13_k_dwarf_oof_predictions.csv


## Interpretation guardrails
- The 3,900–5,200 K range follows Notebook 09; the main Notebook 03 sample uses 4,700–6,500 K.
- Subgroup scores measure classification performance within labeled catalog entries, not planet frequency.
- Small slices and slices with one class do not support stable AUC estimates.
- The catalog-host fraction is not a completeness-corrected occurrence rate and is not calculated here.